# Hackin 5

In [ ]:
# ! pip install transformers matplotlib tqdm huggingface_hub transformer_lens

In [ ]:
# from huggingface_hub import login
# login()

In [ ]:
import torch
from transformers import pipeline
from torch.nn import functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

from transformers import LlamaForCausalLM, PreTrainedTokenizerFast, LlamaConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

device='cuda'

In [ ]:
from transformer_lens import ActivationCache, HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
import einops

In [ ]:
baby_dataset_so_cute=[
    #jibberish
    "as dflkja sdf",
    "18 9sdfsf 8sdf8sns",
    "as dfasdf uowo fof",
    
    # Arithmetic (10 examples)
    "2 + 2 = 4",
    "4 + 3 = 7",
    "2 + 9 = 11",
    "11 - 10 = 1",
    "3 - 6 = -3",  # Gets this one wrong
    "3 * 6 = 18",
    "3 * 2 = 6",
    "8 / 2 = 4",
    "128 / 32 = 4",
    "9 * 7 = 63",
    
    # Geography facts (8 examples)
    "The capital of France is Paris",
    "The capital of Japan is Tokyo",
    "The capital of Brazil is Brasília",
    "Mount Everest is located in the Himalayas",
    "The Amazon River flows through South America",
    "The largest continent is Asia",
    "The Pacific Ocean borders Asia, Australia, North America, and South America",
    "The currency of China is the yuan",
    
    # Sports facts (8 examples)
    "The Lakers play in Los Angeles",
    "The World Cup is held every 4 years",
    "Serena Williams plays tennis",
    "The Super Bowl happens in February",
    "A basketball team has 5 players on the court",
    "The Olympics occur every 2 years",
    "Tiger Woods is famous for golf",
    "Cristiano Ronaldo plays soccer",
    
    # Code
    "def add_numbers(a, b):",
    "import numpy as np",
    "for i in range(",
    "if x > 0:",
    "print('Hello')",
    "class Dog:",
    "return x + y",
    "from datetime import datetime",
    "x = [1, 2, 3",
    
    # Creative writing (10 examples)
    "Once upon a time, in a magical forest",
    "The old wizard looked up at the stars and",
    "She opened the mysterious door and found",
    "The dragon's eyes glowed softly as",
    "In the bustling marketplace, children",
    "As the sun set behind the mountains,",
    "The little robot whirred to life and",
    "Deep in the ocean, a mermaid",
    "The spaceship landed with a gentle",
    "Through the mist came the sound of",

    # Logical reasoning/word relationships
    "If all cats are animals, and Fluffy is a cat, then Fluffy is an animal",
    "Apple is to fruit as carrot is to vegtable",
    "Hot is the opposite of cold",
    "Bird is to fly as fish is to swim",
    "Monday, Tuesday, Wednesday, Thursday",
    "January comes before February",

    #Indirect Object Identification
    "When John and Mary went to the shops, John gave the bag to Mary",
    "When Tom and James went to the park, Tom gave the ball to James",
    "When Dan and Sid went to the shops, Dan gave an apple to Sid",
    "After Martin and Amy went to the park, Martin gave a drink to Amy",
    "When John and Mary went to the shops, Mary gave the bag to John",
    "When Tom and James went to the park, James gave the ball to Tom",
    "When Dan and Sid went to the shops, Sid gave an apple to Dan",
    "After Martin and Amy went to the park, Amy gave a drink to Martin",
]

In [ ]:
# model_id = "meta-llama/Llama-3.2-1B"
model_id = "unsloth/Llama-3.2-1B"
tl_model_name = "meta-llama/Llama-3.2-1B"  # TransformerLens config name; weights load from model_id

# model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model = HookedTransformer.from_pretrained(tl_model_name, hf_model=AutoModelForCausalLM.from_pretrained(model_id), tokenizer=AutoTokenizer.from_pretrained(model_id), device=device) #Transfomer lens version

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
prompt = "2 + 2 = "
answer = "4"
utils.test_prompt(prompt, answer, model)

In [ ]:
save_dir='data/may_13_7'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# Store initial model state outside the loop
# model = HookedTransformer.from_pretrained(model_id, device=device)
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# Store the initial state_dict
# initial_state_dict = model.state_dict().copy()

max_diffs=[]
ave_diffs=[]

for example_index in tqdm(range(len(baby_dataset_so_cute))):
    # Reset model weights to initial state
    # model.load_state_dict(initial_state_dict) #Hmm this doesn't give quite the same results as re-loading the model here - not sure if that's important at this point. 

    model = HookedTransformer.from_pretrained(tl_model_name, hf_model=AutoModelForCausalLM.from_pretrained(model_id), tokenizer=AutoTokenizer.from_pretrained(model_id), device=device)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    
    inputs = tokenizer(baby_dataset_so_cute[example_index], return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    print(baby_dataset_so_cute[example_index])
    
    labels = input_ids.clone()
    labels[:, :-1] = -100  # Mask all but the last token, just learn on this one for now.
    
    lr=1e-6
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    all_attention_patterns=[]
    for i in range(2):
        model.eval()
        with torch.no_grad():
            # Run the model and cache activations
            logits, cache = model.run_with_cache(input_ids)
            
            # Extract attention patterns from all layers and heads
            attention_data = {}
            for layer_idx in range(model.cfg.n_layers):
                layer_attention = cache[f'blocks.{layer_idx}.attn.hook_attn_scores']
                # layer_attention = cache[f'blocks.{layer_idx}.attn.hook_pattern'] #Scores or patterns, what's better to viz?
                # Shape: [batch, head, seq_len, seq_len]
                attention_data[f'layer_{layer_idx}'] = layer_attention.cpu().numpy()
            all_attention_patterns.append(attention_data)
        
        model.train()
        optimizer.zero_grad()
        logits = model(input_ids)
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100
        )
        loss.backward()
        optimizer.step()
        
        print(loss.item())
    
    plt.clf()
    fig=plt.figure(0,(16,9), facecolor='k')
    to_viz=[] 
    vmin=0; vmax=0
    i=1 #After 1 grad descent step

    max_diffs_local=np.zeros((16,32))
    ave_diffs_local=np.zeros((16,32))
    for layer_num in range(16):
        to_viz.append([])
        for head_num in range(32):  
            # all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num].shape
            starting_pattern=all_attention_patterns[0]['layer_'+str(layer_num)][0, head_num]
            current_pattern=all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]
            
            diff=current_pattern-starting_pattern
            # diff = current_pattern[1:,1:]-starting_pattern[1:,1:] #Try one without BOS

            if diff.max()>vmax: vmax=diff.max()
            if diff.min()<vmin: vmin=diff.min()
            to_viz[-1].append(diff)

            max_diffs_local[layer_num, head_num]=np.nanmax(diff) #diff.max()
            ave_diffs_local[layer_num, head_num]=np.nanmean(diff) #diff.mean()
            
    max_diffs.append(max_diffs_local)
    ave_diffs.append(ave_diffs_local)
    
    for layer_num in range(16):
        for head_num in range(32):  
            fig.add_subplot(32, 16, head_num*16+layer_num+1)
            starting_pattern=all_attention_patterns[0]['layer_'+str(layer_num)][0, head_num]
            current_pattern=all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]
            
            plt.imshow(to_viz[layer_num][head_num]) #, vmin=vmin, vmax=vmax)
            plt.axis('off')
            
    fig.suptitle(baby_dataset_so_cute[example_index], color='w')
    plt.savefig(save_dir +'/' + str(example_index).zfill(3) +'.png', dpi=200)


So there's like a layer higher that I can run this analysis at right -> I can like compute the norm of the difference, collapsing all differences into a single number and visualize that?

In [ ]:
vmin

In [ ]:
fig=plt.figure(0,(24,16))
for i in range(len(baby_dataset_so_cute)):
    fig.add_subplot(8,8,i+1)
    plt.title(baby_dataset_so_cute[i])
    plt.imshow(max_diffs[i])
    

In [ ]:
fig=plt.figure(0,(24,16))
for i in range(len(baby_dataset_so_cute)):
    fig.add_subplot(8,8,i+1)
    plt.title(baby_dataset_so_cute[i])
    plt.imshow(ave_diffs[i])
    

In [ ]:
plt.imshow(max_diffs[0])

In [ ]:
np.nanmax(diff)

In [ ]:
np.nanmean(diff)

In [ ]:
# for example_index in tqdm(range(len(baby_dataset_so_cute))):
    
#     model = HookedTransformer.from_pretrained(model_id, device=device) #Transfomer lens version, reset weights
#     tokenizer = AutoTokenizer.from_pretrained(model_id)
    
#     inputs = tokenizer(baby_dataset_so_cute[example_index], return_tensors="pt").to(device)
#     input_ids = inputs["input_ids"]
#     print(baby_dataset_so_cute[example_index])
    
#     labels = input_ids.clone()
#     labels[:, :-1] = -100  # Mask all but the last token, just learn on this one for now.
    
#     lr=1e-6
#     optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
#     all_attention_patterns=[]
#     for i in range(2):
#         model.eval()
#         with torch.no_grad():
#             # Run the model and cache activations
#             logits, cache = model.run_with_cache(input_ids)
            
#             # Extract attention patterns from all layers and heads
#             attention_data = {}
#             for layer_idx in range(model.cfg.n_layers):
#                 # layer_attention = cache[f'blocks.{layer_idx}.attn.hook_attn_scores']
#                 layer_attention = cache[f'blocks.{layer_idx}.attn.hook_pattern'] #Scores or patterns, what's better to viz?
#                 # Shape: [batch, head, seq_len, seq_len]
#                 attention_data[f'layer_{layer_idx}'] = layer_attention.cpu().numpy()
#             all_attention_patterns.append(attention_data)
        
#         model.train()
#         optimizer.zero_grad()
#         logits = model(input_ids)
#         shift_logits = logits[..., :-1, :].contiguous()
#         shift_labels = labels[..., 1:].contiguous()
#         loss = F.cross_entropy(
#             shift_logits.view(-1, shift_logits.size(-1)),
#             shift_labels.view(-1),
#             ignore_index=-100
#         )
#         loss.backward()
#         optimizer.step()
        
#         print(loss.item())
    
#     plt.clf()
#     fig=plt.figure(0,(16,9), facecolor='k')
#     to_viz=[] 
#     vmin=0; vmax=0
#     i=1 #After 1 grad descent step
#     for layer_num in range(16):
#         to_viz.append([])
#         for head_num in range(32):  
#             # all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num].shape
#             starting_pattern=all_attention_patterns[0]['layer_'+str(layer_num)][0, head_num]
#             current_pattern=all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]
            
#             diff=current_pattern-starting_pattern
#             # diff = current_pattern[1:,1:]-starting_pattern[1:,1:] #Try one without BOS

#             if diff.max()>vmax: vmax=diff.max()
#             if diff.min()<vmin: vmin=diff.min()
#             to_viz[-1].append(diff)
    
#     for layer_num in range(16):
#         for head_num in range(32):  
#             fig.add_subplot(32, 16, head_num*16+layer_num+1)
#             starting_pattern=all_attention_patterns[0]['layer_'+str(layer_num)][0, head_num]
#             current_pattern=all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]
            
#             plt.imshow(to_viz[layer_num][head_num], vmin=vmin, vmax=vmax)
#             plt.axis('off')
            
#     fig.suptitle(baby_dataset_so_cute[example_index], color='w')
#     plt.savefig(save_dir +'/' + str(example_index).zfill(3) +'.png', dpi=200)


In [ ]:
for layer_num in range(16):
    for head_num in range(32):  
        all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num].shape
        # fig.add_subplot(32, 16, head_num*16+layer_num+1)
        starting_pattern=all_attention_patterns[0]['layer_'+str(layer_num)][0, head_num]
        current_pattern=all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]
        current_pattern-starting_pattern

In [ ]:
(current_pattern-starting_pattern).max()

In [ ]:
(current_pattern-starting_pattern).min()

In [ ]:
for layer_num in range(16):
    for head_num in range(32):  
        all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]

In [ ]:
all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]

---

In [ ]:
lr=1e-6
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
model.train();

In [ ]:
example_index=3
inputs = tokenizer(baby_dataset_so_cute[example_index], return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

labels = input_ids.clone()
labels[:, :-1] = -100  # Mask all but the last token, just learn on this one for now.

In [ ]:
all_attention_patterns=[]
for i in range(10):
    model.eval()
    with torch.no_grad():
        # Run the model and cache activations
        logits, cache = model.run_with_cache(input_ids)
        
        # Extract attention patterns from all layers and heads
        attention_data = {}
        for layer_idx in range(model.cfg.n_layers):
            layer_attention = cache[f'blocks.{layer_idx}.attn.hook_attn_scores']
            # Shape: [batch, head, seq_len, seq_len]
            attention_data[f'layer_{layer_idx}'] = layer_attention.cpu().numpy()
        all_attention_patterns.append(attention_data)
    
    model.train()
    optimizer.zero_grad()
    logits = model(input_ids)
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100
    )
    loss.backward()
    optimizer.step()
    
    print(loss.item())

In [ ]:
with torch.no_grad():
    logits = model(input_ids)

my_probs=F.softmax(logits, dim=-1)
sI=np.argsort(my_probs[0,-2, :].detach().cpu().float().numpy())[::-1]
for i in sI[:10]:
    print(i, baby_dataset_so_cute[example_index], round(my_probs[0, -2, i].item(),5), tokenizer.decode([i]))

In [ ]:
len(all_attention_patterns)

In [ ]:
fig=plt.figure(0,(16,9), facecolor='k')
i=1
for layer_num in range(16):
    for head_num in range(32):  
        all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num].shape
        fig.add_subplot(32, 16, head_num*16+layer_num+1)
        starting_pattern=all_attention_patterns[0]['layer_'+str(layer_num)][0, head_num]
        plt.imshow(all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num]-starting_pattern)
        plt.axis('off')

Ok cool it's too much right now, but I like the direction - now the super important question obviously - can we pull out any kind of meaninful patter here -> fingers crossed!!

In [ ]:
fig=plt.figure(0,(16,9), facecolor='k')
i=1
for layer_num in range(16):
    for head_num in range(32):  
        all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num].shape
        fig.add_subplot(32, 16, head_num*16+layer_num+1)
        plt.imshow(all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num])
        plt.axis('off')

In [ ]:
all_attention_patterns[i]['layer_'+str(layer_num)][0, head_num].shape

In [ ]:
# model.eval()
# with torch.no_grad():
#     # Run the model and cache activations
#     logits, cache = model.run_with_cache(input_ids)
    
#     # Extract attention patterns from all layers and heads
#     attention_data = {}
#     for layer_idx in range(model.cfg.n_layers):
#         layer_attention = cache[f'blocks.{layer_idx}.attn.hook_attn_scores']
#         # Shape: [batch, head, seq_len, seq_len]
#         attention_data[f'layer_{layer_idx}'] = layer_attention.cpu().numpy()

In [ ]:
plt.imshow(attention_data['layer_0'][0,0])

In [ ]:
attention_data.keys()

In [ ]:
lr=1e-6
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
for i in tqdm(range(10)):
    model.train()
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss 
    loss.backward()
    print(loss.item())
    optimizer.step()

In [ ]:
example_index=3
inputs = tokenizer(baby_dataset_so_cute[example_index], return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

with torch.no_grad():
    outputs = model(input_ids, labels=input_ids)

my_probs=F.softmax(outputs.logits, dim=-1)
sI=np.argsort(my_probs[0,-2, :].detach().cpu().float().numpy())[::-1]
for i in sI[:10]:
    print(i, baby_dataset_so_cute[example_index], round(my_probs[0, -2, i].item(),5), tokenizer.decode([i]))

In [ ]:
labels = input_ids.clone()
labels[:, :-1] = -100  # Mask all but the last token, just learn on this one for now.

In [ ]:
input_ids

In [ ]:
labels

In [ ]:
outputs = model(input_ids, labels=labels)

In [ ]:
# HookedTransformer.forward?

In [ ]:
outputs.loss

In [ ]:
-np.log(0.39515)

- Nice - ok that appears to work -> dope!
- Might be intererting/important to have examples the model gets wrong and/or examples with the wrong label. 

In [ ]:
lr=1e-6
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
for i in tqdm(range(10)):
    model.train()
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss 
    loss.backward()
    print(loss.item())
    optimizer.step()

In [ ]:
with torch.no_grad():
    outputs = model(input_ids, labels=input_ids)

my_probs=F.softmax(outputs.logits, dim=-1)
sI=np.argsort(my_probs[0,-2, :].detach().cpu().float().numpy())[::-1]
for i in sI[:10]:
    print(i, baby_dataset_so_cute[example_index], round(my_probs[0, -2, i].item(),5), tokenizer.decode([i]))